# **0 Install Library**

In [1]:
!pip install climada
!pip install fiona

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 778.9/778.9 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7

# **1. Using Random Exposure Data + from_calibrated_regional_ImpfSet() with id = 7 + ERA5LAND Wind Gust CSV to find the annual average impact**

In [38]:
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy.interpolate import griddata
from scipy.sparse import csr_matrix
import os
from climada.entity import Exposures, ImpactFuncSet, IFTropCyclone
from climada.entity.impact_funcs import ImpactFuncSet, ImpfTropCyclone
from climada.hazard import Centroids, TCTracks, TropCyclone
from climada.engine import ImpactCalc
from climada.engine import Impact
from climada.entity import LitPop
from climada.entity.impact_funcs.trop_cyclone import ImpfSetTropCyclone
import matplotlib.pyplot as plt

In [62]:
import geopandas as gdf

exp_lp = gdf.read_file("Random_Exposure_PHL_withValue.shp")

min_lat, max_lat, min_lon, max_lon = exp_lp.total_bounds  # Use bounds from exposure data
cent = Centroids.from_pnt_bounds((min_lon, min_lat, max_lon, max_lat), res=0.01)

exp_lp = Exposures(exp_lp)
print(exp_lp)

description: None
ref_year: 2018
value_unit: USD
crs: EPSG:4326
data: (1000 entries)
     rand_point iso3        status color_code         name continent  \
0           0.0  PHL  Member State        PHL  Philippines      Asia   
1           1.0  PHL  Member State        PHL  Philippines      Asia   
2           2.0  PHL  Member State        PHL  Philippines      Asia   
3           3.0  PHL  Member State        PHL  Philippines      Asia   
996       996.0  PHL  Member State        PHL  Philippines      Asia   
997       997.0  PHL  Member State        PHL  Philippines      Asia   
998       998.0  PHL  Member State        PHL  Philippines      Asia   
999       999.0  PHL  Member State        PHL  Philippines      Asia   

                 region iso_3166_1   french_sho ExposureVa  Exposure_1  \
0    South-Eastern Asia         PH  Philippines       None     10000.0   
1    South-Eastern Asia         PH  Philippines       None     10000.0   
2    South-Eastern Asia         PH  Philippi

In [63]:
# Initialize list to store impacts
all_impacts = []

In [66]:
# Step 4: Process each CSV file
import glob

csv_files = glob.glob("./WindGustCSV/max_wind_gust_*.csv")  # Adjust path as needed
if len(csv_files) != 445:
    print(f"Warning: Found {len(csv_files)} CSV files, expected 445.")

for csv_file in csv_files:
    # Extract filename for reporting
    filename = os.path.basename(csv_file)

    # Load wind gust data
    wind_gust_df = pd.read_csv(csv_file)

    # Verify columns
    expected_columns = ['latitude', 'longitude', 'max_wind_gust']
    if not all(col in wind_gust_df.columns for col in expected_columns):
        print(f"Skipping {csv_file}: Missing required columns.")
        continue

    wind_gust_points = wind_gust_df[['latitude', 'longitude']].values
    wind_gust_values = wind_gust_df['max_wind_gust'].values  # Assumed in m/s

    # Interpolate wind gust data to centroids
    cent_coords = np.stack([cent.lat, cent.lon], axis=1)
    wind_gust_interp = griddata(
        wind_gust_points,
        wind_gust_values,
        cent_coords,
        method='nearest'
    )

    # Verify interpolated data shape
    n_centroids = cent.size
    if wind_gust_interp.shape[0] != n_centroids:
        print(f"Warning: Interpolated data size mismatch for {filename}. Skipping.")
        continue


    # Set up tropical cyclone hazard
    tc_pnt = TropCyclone()
    tc_pnt.centroids = cent
    tc_pnt.event_id = np.array([1])
    tc_pnt.event_name = [filename.split('_')[3]]
    tc_pnt.frequency = np.array([1.0])
    tc_pnt.haz_type = 'TC'

    # Initialize intensity and fraction matrices
    tc_pnt.intensity = csr_matrix((1, n_centroids), dtype=np.float64)
    tc_pnt.intensity[0, :] = wind_gust_interp
    fraction_data = (wind_gust_interp > 0).astype(np.float64)
    tc_pnt.fraction = csr_matrix(fraction_data.reshape(1, n_centroids), dtype=np.float64)

    tc_pnt.check()

    # impact function for Philippines
    impf_pnt = ImpactFuncSet()
    imp_fun_set_TC = ImpfSetTropCyclone.from_calibrated_regional_ImpfSet()
    impf_phl = imp_fun_set_TC.get_func(haz_type="TC", fun_id=7)  # ID 7 for Philippines
    impf_pnt.append(impf_phl)
    impf_pnt.check()

    haz_type = "TC"
    haz_id = 7  # Philippines-specific impact function

    # Rename 'Exposure_1' to 'value'
    exp_lp.gdf.rename(columns={"Exposure_1": "value"}, inplace=True)
    exp_lp.gdf.rename(columns={"impf_": f"impf_{haz_type}"}, inplace=True)
    exp_lp.gdf[f"impf_{haz_type}"] = haz_id
    exp_lp.check()

    # Compute economic impact
    imp_pnt = ImpactCalc(exp_lp, impf_pnt, tc_pnt).impact(save_mat=True)

    print(f"Total Economic Impact (USD): {imp_pnt.aai_agg:,.2f}")
    print(f"Impact per event (USD): {imp_pnt.at_event[0]:,.2f}")
    print(f"Sum of the direct impact at all exposure point: : {imp_pnt.eai_exp.sum():,.2f}")
    print(f"total exposure value affected (USD): {imp_pnt.tot_value.sum():,.2f}")

    # Collect results
    all_impacts.append({
        'filename': csv_file,
        'aai_agg': imp_pnt.aai_agg,
        'at_event': imp_pnt.at_event[0],
        'eai_exp_sum': imp_pnt.eai_exp.sum(),
        'tot_value_sum': imp_pnt.tot_value.sum()
    })

    print(f"add the result to all_impacts of {csv_file}")

2025-08-06 06:44:17,216 - climada.engine.impact_calc - WARNING - No exposures with value >0 in the vicinity of the hazard.
Total Economic Impact (USD): 0.00
Impact per event (USD): 0.00
Sum of the direct impact at all exposure point: : 0.00
2025-08-06 06:44:17,219 - climada.engine.impact - WARNING - The Impact.tot_value attribute is deprecated.Use Exposures.affected_total_value to calculate the affected total exposure value based on a specific hazard intensity threshold
total exposure value affected (USD): 0.00
2025-08-06 06:44:17,220 - climada.engine.impact - WARNING - The Impact.tot_value attribute is deprecated.Use Exposures.affected_total_value to calculate the affected total exposure value based on a specific hazard intensity threshold
add the result to all_impacts of ./WindGustCSV/max_wind_gust_2011126N11129.csv
2025-08-06 06:44:17,243 - climada.engine.impact_calc - WARNING - No exposures with value >0 in the vicinity of the hazard.
Total Economic Impact (USD): 0.00
Impact per ev

In [50]:
# Step 8: Save results to CSV
results_df = pd.DataFrame(all_impacts)
results_df.to_csv("tc_impact_results.csv", index=False)
